# 4. Query the Pipeline

This notebook demonstrates DataJoint query operations on the LC-MS pipeline.

In [ ]:
from lcms_demo.config import use_local_database

use_local_database()

In [ ]:
from lcms_demo import subject, session, scan

## Basic Queries

### Fetch All Data

In [ ]:
# Fetch all subjects as a pandas DataFrame
subjects_df = subject.Subject.fetch(format="frame")
subjects_df

In [ ]:
# Fetch as list of dictionaries
subject.Subject.fetch(as_dict=True)

### Restriction (Filtering)

In [ ]:
# Filter by exact match
subject.Subject & {"subject_id": "SUBJ_001"}

In [ ]:
# Filter samples by type
subject.Sample & "sample_type = 'plasma'"

In [ ]:
# Filter by pattern matching
subject.Subject & "subject_id LIKE 'SUBJ_%'"

### Projection (Selecting Columns)

In [ ]:
# Select specific attributes
scan.Scan.proj("retention_time", "total_ion_current")

In [ ]:
# Rename attributes
scan.Scan.proj(rt="retention_time", tic="total_ion_current")

In [ ]:
# Compute new attributes
scan.Scan.proj(rt_seconds="retention_time * 60")

### Join Operations

In [ ]:
# Join tables to get combined information
subject.Subject * subject.Sample

In [ ]:
# Join with restriction
(subject.Subject * subject.Sample) & "sample_type = 'plasma'"

### Aggregation

In [ ]:
# Count scans per session
scan.Scan.aggr(session.Session, n_scans="count(*)")

In [ ]:
# Average TIC per session
scan.Scan.aggr(session.Session, avg_tic="avg(total_ion_current)")

## Advanced Queries

In [ ]:
# Find sessions with high TIC scans
high_tic_scans = scan.Scan & "total_ion_current > 1e6"
high_tic_scans

In [ ]:
# Get scans in a retention time window
rt_window = scan.Scan & "retention_time BETWEEN 5 AND 10"
rt_window

In [ ]:
# Complex query: plasma samples with their sessions
plasma_sessions = (
    session.Session
    * (subject.Sample & "sample_type = 'plasma'")
)
plasma_sessions

## Fetch Options

In [ ]:
# Fetch with ORDER BY and LIMIT
scan.Scan.fetch(
    "scan_number", "retention_time",
    order_by="retention_time DESC",
    limit=10,
    format="frame"
)

In [ ]:
# Fetch single row
first_scan = (scan.Scan & "scan_number = 1").fetch1()

In [ ]:
# Check if query returns any results
has_scans = bool(scan.Scan)
print(f"Has scans: {has_scans}")
print(f"Number of scans: {len(scan.Scan())}")